In [2]:
import pandas as pd

# 1️⃣ CSV 로드 + 필터링
df = pd.read_csv("data/1_2_(로우데이터_합본.csv)목적별_국적별_입국(05년1월~25년5월).csv", encoding='cp949')

df = df[
    (~df['국적'].str.endswith('주', na=False)) &
    (~df['국적'].str.endswith('기타', na=False)) &
    (~df['국적'].isin(['소 계', '전 체', '미상', '기 타', '교포', '중 동'])) &
    (~df['목적'].isin(['소 계', '전 체']))
]
df = df[df['국적'].notna()]

# 2️⃣ 컬럼 분류
fixed_cols = ['국적', '목적']
date_cols = [col for col in df.columns if ('년' in col and '월' in col)]

# ✅ 범위 제외: (예시) 마지막 2개 열 제외
date_cols_final = date_cols[:-2]
cols_to_use = fixed_cols + date_cols_final
df = df[cols_to_use]

# 3️⃣ 날짜 컬럼 숫자화 (쉼표 제거)
for col in date_cols_final:
    df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# 4️⃣ long-form 변환
long_df = df.melt(id_vars=fixed_cols, value_vars=date_cols_final,
                  var_name='ym', value_name='입국자수')

# 5️⃣ 'ym'에서 년, 월 추출
long_df['년'] = long_df['ym'].str.extract(r'(\d{4})').astype(int)
long_df['월'] = long_df['ym'].str.extract(r'(\d{1,2})월').astype(int)

# 6️⃣ 컬럼 정렬 및 불필요 데이터 제거
long_df = long_df[['국적', '목적', '년', '월', '입국자수']]
long_df = long_df[long_df['입국자수'].notnull() & (long_df['입국자수'] >= 0)]

# 7️⃣ 저장
long_df.to_csv("data/목적별국적별입국소계제거.csv", index=False, encoding='cp949')
